In [8]:
import os
from SelfCal import PipelineWrapper
from SelfCal.MakeMap import load_reproj_file

from astropy.io.votable import parse, parse_single_table
from astropy.io import fits
from astropy.wcs import WCS
import glob
from tqdm import tqdm
import h5py
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
import matplotlib as mpl
mpl.rcParams['figure.dpi'] = 200


In [14]:
exp_dir = '/data1/Euclid/OTF/NISP/'
exposure_list = glob.glob(exp_dir + '*_Y*.fits')
dq_ext_list = (np.arange(0,16)*3+3)
sci_ext_list = (np.arange(0,16)*3+1)


In [13]:
config = {}
config['output_dir'] = '/mnt/md124/thomasli/selfcal/outputs/'
config['run_name'] = f'EDFN_Y_3p0arcsec'
config['resolution_arcsec'] = 3

In [15]:
rr = PipelineWrapper.Reprojector(config, exposure_list=exposure_list)
rr.define_reference(padding_pixels=100, use_ext=[1, 10, 37, 46])

Reference WCS not found at /mnt/md124/thomasli/selfcal/outputs/EDFN_Y_3p0arcsec/ref.fits. Creating a new reference frame.
Defining optimal celestial WCS...


Loading corner WCS: 100%|█████████████████████████████████████████████████████████████████████████████████████████| 904/904 [34:08<00:00,  2.27s/it]


Reference frame FITS saved to: /mnt/md124/thomasli/selfcal/outputs/EDFN_Y_3p0arcsec/ref.fits
Reference WCS saved to /mnt/md124/thomasli/selfcal/outputs/EDFN_Y_3p0arcsec/ref.fits
Mosaic shape: (7843, 7657)
Mosaic WCS: WCS Keywords

Number of WCS axes: 2
CTYPE : 'RA---TAN' 'DEC--TAN' 
CRVAL : 269.84699816649936 65.98859911120327 
CRPIX : 3977.0777900373314 3893.8517817139937 
PC1_1 PC1_2  : 1.0 0.0 
PC2_1 PC2_2  : 0.0 1.0 
CDELT : -0.0008333333333333333 0.0008333333333333333 
NAXIS : 0  0


In [16]:
rr.run_reproject(max_workers=150, reproj_func='exact', padding_percentage=0.05, 
                    sci_ext_list=sci_ext_list, 
                    dq_ext_list=dq_ext_list, 
                    exp_idx_list=np.arange(0, len(exposure_list)), 
                    det_idx_list=np.arange(0,16),
                    replace_existing=False,
                 reproject_kwargs={'parallel': 2}
                )

Starting batch reprojection. Output will be saved to: /mnt/md124/thomasli/selfcal/outputs/EDFN_Y_3p0arcsec/reprojected


Reprojecting frames:  62%|████████████████████████████████████████████████████▊                                | 8979/14464 [20:46<14:15,  6.41it/s]

Error processing detector 7 from exposure file index 562 (/data1/Euclid/OTF/NISP/EUC_NIR_W-CAL-IMAGE_Y-2057-1_20240706T090103.710296Z.fits): ERROR 8 in wcss2p() at line 3966 of file cextern/wcslib/C/wcs.c:
One or more of the pixel coordinates were invalid.
ERROR 6 in linx2p() at line 979 of file cextern/wcslib/C/lin.c:
De-distort error.
ERROR 5 in disx2p() at line 1411 of file cextern/wcslib/C/dis.c:
Convergence not achieved after 30 iterations, residual 2.7e-06.



Reprojecting frames:  91%|████████████████████████████████████████████████████████████████████████████▋       | 13209/14464 [30:56<03:57,  5.29it/s]

Error processing detector 10 from exposure file index 825 (/data1/Euclid/OTF/NISP/EUC_NIR_W-CAL-IMAGE_Y-2205-1_20240716T153922.848137Z.fits): ERROR 8 in wcss2p() at line 3966 of file cextern/wcslib/C/wcs.c:
One or more of the pixel coordinates were invalid.
ERROR 6 in linx2p() at line 979 of file cextern/wcslib/C/lin.c:
De-distort error.
ERROR 5 in disx2p() at line 1411 of file cextern/wcslib/C/dis.c:
Convergence not achieved after 30 iterations, residual 2.8e-10.



Reprojecting frames: 100%|████████████████████████████████████████████████████████████████████████████████████| 14464/14464 [33:48<00:00,  7.13it/s]


Batch reprojection completed. 14462 frames successfully processed out of 14464.
Reprojection completed in 2030.34 seconds.


In [43]:
calib_set = sorted(glob.glob('/mnt/md124/thomasli/selfcal/outputs/EDFN_Y_3p0arcsec/reprojected/*.h5'))
coadd_set = sorted(glob.glob('/mnt/md124/thomasli/selfcal/outputs/EDFN_Y_0p3arcsec/reprojected/*.h5'))

calib_set_basenames = [os.path.basename(f) for f in calib_set]
coadd_set_basenames = [os.path.basename(f) for f in coadd_set]
disjoint = list(set(calib_set_basenames) ^ set(coadd_set_basenames))



In [42]:
for f in calib_set:
    if os.path.basename(f) in disjoint:
        print(f"File {os.path.basename(f)} is disjoint, removing file.")
        os.remove(f)

for f in coadd_set:
    if os.path.basename(f) in disjoint:
        print(f"File {os.path.basename(f)} is disjoint, removing file.")
        os.remove(f)

File exp_0002_det_06.h5 is disjoint, removing file.
File exp_0003_det_11.h5 is disjoint, removing file.
File exp_0006_det_04.h5 is disjoint, removing file.
File exp_0009_det_09.h5 is disjoint, removing file.
File exp_0011_det_07.h5 is disjoint, removing file.
File exp_0014_det_11.h5 is disjoint, removing file.
File exp_0033_det_11.h5 is disjoint, removing file.
File exp_0034_det_04.h5 is disjoint, removing file.
File exp_0045_det_01.h5 is disjoint, removing file.
File exp_0048_det_05.h5 is disjoint, removing file.
File exp_0048_det_10.h5 is disjoint, removing file.
File exp_0055_det_15.h5 is disjoint, removing file.
File exp_0066_det_08.h5 is disjoint, removing file.
File exp_0068_det_09.h5 is disjoint, removing file.
File exp_0076_det_11.h5 is disjoint, removing file.
File exp_0082_det_04.h5 is disjoint, removing file.
File exp_0085_det_07.h5 is disjoint, removing file.
File exp_0092_det_02.h5 is disjoint, removing file.
File exp_0092_det_11.h5 is disjoint, removing file.
File exp_009

In [17]:
rr.get_reproj_files()
reproj_list = sorted(rr.reproj_list)
bad_list = []
for f in tqdm(reproj_list):
    try:
        with h5py.File(f, 'r') as hf:
            continue
    except OSError:
        print(f'Found corrupted file {f}')
        bad_list.append(f)

  0%|                                                                                                                     | 0/14462 [00:00<?, ?it/s]

100%|███████████████████████████████████████████████████████████████████████████████████████████████████████| 14462/14462 [00:01<00:00, 8642.33it/s]


In [14]:
for f in bad_list:
    os.remove(f)

In [47]:
def compute_chunk_edges(det_shape, chunk_size):
    '''Split detector into chunks and return edge of chunks'''
    det_h, det_w = det_shape
    chunk_h, chunk_w = chunk_size
    y_edges = np.arange(0, det_h + 1, chunk_h)
    x_edges = np.arange(0, det_w + 1, chunk_w)
    
    return x_edges, y_edges

x_edges, y_edges = compute_chunk_edges((2040, 2040), (204, 204))
chunk_map = np.zeros((2040, 2040), dtype=int)

chunk_id = 0

for j in range(len(y_edges)-1):
    y_low = y_edges[j]
    y_high = y_edges[j+1]
    for i in range(len(x_edges)-1):
        x_low = x_edges[i]
        x_high = x_edges[i+1]
        chunk_map[y_low:y_high, x_low:x_high] = chunk_id

        chunk_id += 1

In [51]:
cc = PipelineWrapper.Calibrator(config)

100%|████████████████████████████████████████████████████████████████████████████████████████████████████| 14308/14308 [00:00<00:00, 1593608.31it/s]

Loading reference frame from: /mnt/md124/thomasli/selfcal/outputs/EDFN_Y_3p0arcsec/ref.fits


In [11]:
# import importlib
# import SelfCal.MakeMap
# importlib.reload(SelfCal.MakeMap)
# from SelfCal.MakeMap import _prep_subframe
# ref_coords, sub_data, sub_weight, chunk_contrib, sub_aux = _prep_subframe(cc.reproj_list[0], chunk_map, apply_weight=False, apply_mask=True, 
#                    chunk_offset=None, det_offset_func=None, ignore_list=[11,15], 
#                    det_valid_mask=None, valid_threshold=0.99, 
#                    for_lsqr=True, oversample_factor=1, 
#                    # These arguments are accepted for compatibility/internal logic 
#                    # but might not be used depending on logic path
#                    exp_idx=None, det_idx=None, det_aux=None)

In [52]:
# cc.reproj_list = cc.reproj_list[0:500]
# cc.exp_idx_list = cc.exp_idx_list[0:500]
# cc.det_idx_list = cc.det_idx_list[0:500]

In [53]:
cc.setup_lsqr(
    apply_mask=True, 
    apply_weight=False, 
    chunk_map=chunk_map, 
    det_valid_mask=np.ones((2040, 2040)),
    max_workers=50, 
    outlier_thresh=10.0,
    ignore_list=[11,15],
    oversample_factor=1,
    batch_size=100
    )

Processing 14308 items in 144 batches (Batch Size: 100)...


Building A, b matrix: 100%|███████████████████████████████████████████████████████████████████████████████████████| 144/144 [01:03<00:00,  2.27it/s]


LSQR setup completed in 82.55 seconds.


In [55]:
cc.apply_lsqr(x0=None, atol=1e-06, btol=1e-06, damp=1e-3, iter_lim=200)

Solving least squares for 60144251 unknowns with 465991945 equations.
 
LSQR            Least-squares solution of  Ax = b
The matrix A has 465991945 rows and 60144251 columns
damp = 1.00000000000000e-03   calc_var =        0
atol = 1.00e-06                 conlim = 1.00e+08
btol = 1.00e-06               iter_lim =      200
 
   Itn      x[0]       r1norm     r2norm   Compatible    LS      Norm A   Cond A
     0  0.00000e+00   1.415e+06  1.415e+06    1.0e+00  5.0e-05
     1  0.00000e+00   1.545e+05  1.545e+05    1.1e-01  3.8e-01   7.1e+01  1.0e+00
     2  0.00000e+00   1.426e+05  1.426e+05    1.0e-01  2.8e-02   1.0e+02  2.0e+00
     3  0.00000e+00   1.422e+05  1.422e+05    1.0e-01  3.4e-02   1.1e+02  3.3e+00
     4  0.00000e+00   1.286e+05  1.286e+05    9.1e-02  1.9e-01   1.2e+02  2.3e+01
     5  0.00000e+00   1.130e+05  1.130e+05    8.0e-02  3.0e-02   1.4e+02  3.8e+01
     6  0.00000e+00   1.127e+05  1.127e+05    8.0e-02  5.6e-03   1.6e+02  4.2e+01
     7  0.00000e+00   1.127e+05  1.12

In [ ]:
cc.O.shape

array([[63.51768868, 62.93531829, 62.96378106, ..., 62.83089966,
        62.69782878, 63.07218571],
       [63.74742099, 63.01353793, 63.32351996, ..., 63.80003622,
        63.69608915, 63.86572173],
       [63.88406814, 62.96151727, 63.34226972, ..., 63.91331314,
        63.73676907, 64.27314295],
       ...,
       [63.81708851, 63.03291772, 63.01850601, ..., 62.31795943,
        62.55241967, 62.89416193],
       [63.43301792, 62.6871011 , 62.61868874, ..., 63.01308263,
        62.50337525, 63.08915426],
       [71.78336413, 71.03824062, 71.11329524, ..., 72.43267496,
        71.84839297, 71.72495096]], shape=(904, 100))

In [ ]:
cc.save_calibration

In [31]:
import gc

In [34]:
del rr
gc.collect()

680

In [30]:
A = cc.A.copy()

In [28]:
print(str(cc.A.data.nbytes/(1024**3))+'GB')
print(str(cc.A.coords[0].nbytes/(1024**3))+'GB')
print(str(cc.A.coords[1].nbytes/(1024**3))+'GB')

13.078589301556349GB
26.157178603112698GB
26.157178603112698GB
